In [1]:
import csv

with open("merged_fraud_dataset.csv", "r", encoding="utf-8") as file:
    reader = csv.reader(file)
    header = next(reader)

print("Number of columns:", len(header))
print("Columns:")
print(header)

Number of columns: 45
Columns:
['transaction_id', 'transaction_date', 'customer_id', 'card_type', 'amount', 'merchant_category', 'merchant_country', 'transaction_channel', 'device_type', 'is_international', 'is_card_present', 'transactions_last_24h', 'average_amount_30d', 'risk_score', 'is_fraud', 'transaction_hour', 'transaction_day', 'transaction_month', 'dataset_source', 'sender_account_id', 'receiver_account_id', 'transaction_type', 'sender_balance_before', 'sender_balance_after', 'receiver_balance_before', 'receiver_balance_after', 'customer_age_group', 'customer_city', 'kyc_level', 'login_attempts', 'account_age_days', 'product_category', 'payment_method', 'billing_country', 'shipping_country', 'email_age_days', 'ip_risk_level', 'failed_payment_attempts', 'delivery_speed', 'customer_balance', 'location', 'distance_from_home_km', 'transactions_last_1h', 'new_device', 'new_location']


In [2]:
import csv

with open("merged_fraud_dataset.csv", "r", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    data = list(reader)

print("Total records:", len(data))
print("Total columns:", len(data[0]))
print("First record:")
print(data[0])

Total records: 57661
Total columns: 45
First record:
{'transaction_id': 'CC00011340', 'transaction_date': '2025-07-15 19:54:24', 'customer_id': 'C002995', 'card_type': 'visa', 'amount': '55.82', 'merchant_category': 'travel', 'merchant_country': 'uae', 'transaction_channel': 'website', 'device_type': 'windows', 'is_international': '0.0', 'is_card_present': '1.0', 'transactions_last_24h': '7.0', 'average_amount_30d': '77.01', 'risk_score': '10.0', 'is_fraud': '0', 'transaction_hour': '19', 'transaction_day': '15', 'transaction_month': '7', 'dataset_source': 'Credit Card Fraud', 'sender_account_id': '', 'receiver_account_id': '', 'transaction_type': '', 'sender_balance_before': '', 'sender_balance_after': '', 'receiver_balance_before': '', 'receiver_balance_after': '', 'customer_age_group': '', 'customer_city': '', 'kyc_level': '', 'login_attempts': '', 'account_age_days': '', 'product_category': '', 'payment_method': '', 'billing_country': '', 'shipping_country': '', 'email_age_days': '

In [3]:
from collections import Counter

fraud_counts = Counter(row["is_fraud"] for row in data)

print("is_fraud distribution:")
for value, count in fraud_counts.items():
    print(value, ":", count)

is_fraud distribution:
0 : 56663
1 : 998


In [4]:
import random

# Make a copy of the data
all_data = data.copy()

# Separate fraud and non-fraud transactions
non_fraud = [row for row in all_data if row["is_fraud"] == "0"]
fraud = [row for row in all_data if row["is_fraud"] == "1"]

# Shuffle each class
random.seed(42)
random.shuffle(non_fraud)
random.shuffle(fraud)

# Function to split each class into 70%, 15%, 15%
def split_class(rows):
    n = len(rows)
    
    train_end = int(n * 0.70)
    validation_end = train_end + int(n * 0.15)
    
    train = rows[:train_end]
    validation = rows[train_end:validation_end]
    test = rows[validation_end:]
    
    return train, validation, test

# Split both classes
non_fraud_train, non_fraud_val, non_fraud_test = split_class(non_fraud)
fraud_train, fraud_val, fraud_test = split_class(fraud)

# Combine the classes
train_data = non_fraud_train + fraud_train
validation_data = non_fraud_val + fraud_val
test_data = non_fraud_test + fraud_test

# Shuffle the final sets
random.shuffle(train_data)
random.shuffle(validation_data)
random.shuffle(test_data)

print("Training set:", len(train_data))
print("Validation set:", len(validation_data))
print("Test set:", len(test_data))
print("Total:", len(train_data) + len(validation_data) + len(test_data))

Training set: 40362
Validation set: 8648
Test set: 8651
Total: 57661


In [5]:
from collections import Counter

def show_distribution(name, dataset):
    counts = Counter(row["is_fraud"] for row in dataset)
    total = len(dataset)
    
    non_fraud_count = counts.get("0", 0)
    fraud_count = counts.get("1", 0)
    
    print(f"\n{name}")
    print("-" * 30)
    print("Total:", total)
    print("Non-Fraud (0):", non_fraud_count)
    print("Fraud (1):", fraud_count)
    print("Fraud percentage:", round((fraud_count / total) * 100, 2), "%")

show_distribution("TRAINING SET", train_data)
show_distribution("VALIDATION SET", validation_data)
show_distribution("TEST SET", test_data)


TRAINING SET
------------------------------
Total: 40362
Non-Fraud (0): 39664
Fraud (1): 698
Fraud percentage: 1.73 %

VALIDATION SET
------------------------------
Total: 8648
Non-Fraud (0): 8499
Fraud (1): 149
Fraud percentage: 1.72 %

TEST SET
------------------------------
Total: 8651
Non-Fraud (0): 8500
Fraud (1): 151
Fraud percentage: 1.75 %


In [6]:
import csv

# Save training dataset
with open("train_fraud.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=data[0].keys())
    writer.writeheader()
    writer.writerows(train_data)

# Save validation dataset
with open("validation_fraud.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=data[0].keys())
    writer.writeheader()
    writer.writerows(validation_data)

# Save test dataset
with open("test_fraud.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=data[0].keys())
    writer.writeheader()
    writer.writerows(test_data)

print("✅ All three datasets saved successfully!")
print("1. train_fraud.csv")
print("2. validation_fraud.csv")
print("3. test_fraud.csv")

✅ All three datasets saved successfully!
1. train_fraud.csv
2. validation_fraud.csv
3. test_fraud.csv


In [7]:
import csv

files = {
    "train_fraud.csv": 40362,
    "validation_fraud.csv": 8648,
    "test_fraud.csv": 8651
}

for filename, expected_rows in files.items():
    with open(filename, "r", encoding="utf-8") as file:
        reader = csv.reader(file)
        rows = list(reader)

    actual_rows = len(rows) - 1  # subtract header

    print(f"{filename}")
    print(f"Expected rows: {expected_rows}")
    print(f"Actual rows:   {actual_rows}")

    if actual_rows == expected_rows:
        print("✅ Correct")
    else:
        print("❌ Row count mismatch")

    print("-" * 40)

train_fraud.csv
Expected rows: 40362
Actual rows:   40362
✅ Correct
----------------------------------------
validation_fraud.csv
Expected rows: 8648
Actual rows:   8648
✅ Correct
----------------------------------------
test_fraud.csv
Expected rows: 8651
Actual rows:   8651
✅ Correct
----------------------------------------
